# 01 · Exploring MIND and choosing the slider buckets

OwnFeed lets people steer their feed with 5 sliders. Before building anything, we need to answer:

1. What kinds of articles are in MIND, and which ones do people actually click?
2. How should the ~17 MIND categories be grouped into a few slider buckets?
3. Do different users have different tastes? (If everyone liked the same mix, there would be nothing to steer.)
4. Is the mix stable from day to day? (This matters for the time-based train/validation split.)

We only use the **train** data here. The test folder is locked until the final results.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

DATA = Path("../data/MINDsmall_train")
FIGURES = Path("../results/figures")
FIGURES.mkdir(parents=True, exist_ok=True)

## Load the data

- `news.tsv`: one row per article (id, category, subcategory, title, abstract, ...)
- `behaviors.tsv`: one row per **impression**, i.e. one time a user opened the page. It lists the articles shown and whether each one was clicked (`N123-1` = clicked, `N123-0` = skipped).

In [ ]:
news = pd.read_csv(
    DATA / "news.tsv", sep="\t", header=None, quoting=3,
    names=["news_id", "category", "subcategory", "title", "abstract", "url", "title_entities", "abstract_entities"],
)
behaviors = pd.read_csv(
    DATA / "behaviors.tsv", sep="\t", header=None,
    names=["impression_id", "user_id", "time", "history", "impressions"],
)
behaviors["time"] = pd.to_datetime(behaviors["time"], format="%m/%d/%Y %I:%M:%S %p")

print(f"{len(news):,} articles, {len(behaviors):,} impressions, {behaviors.user_id.nunique():,} users")
print("Time range:", behaviors.time.min(), "to", behaviors.time.max())
news.head(3)

## Step 1: How many articles are in each category?

In [ ]:
news["category"].value_counts()

## Step 2: Which categories do people actually click?

Article counts can mislead: lots of articles doesn't mean lots of interest. So we unpack every impression into one row per *(article shown, clicked or not)* and count clicks.

In [ ]:
shown = behaviors[["impression_id", "user_id", "time", "impressions"]].copy()
shown["item"] = shown["impressions"].str.split()
shown = shown.explode("item").drop(columns="impressions")
shown[["news_id", "clicked"]] = shown["item"].str.split("-", expand=True)
shown["clicked"] = shown["clicked"].astype(int)
shown = shown.drop(columns="item").merge(news[["news_id", "category", "subcategory"]], on="news_id", how="left")

by_category = shown.groupby("category").agg(shown=("clicked", "size"), clicks=("clicked", "sum"))
by_category["click_share_%"] = (100 * by_category.clicks / by_category.clicks.sum()).round(1)
by_category["ctr_%"] = (100 * by_category.clicks / by_category.shown).round(2)
by_category.sort_values("clicks", ascending=False)

## Step 2b: Categories and their subcategories

Each category is split into finer **subcategories**. `news`, for example, contains US news, crime, politics, world, science and more. This tells us what is really inside each category before we group them.

First an overview: for each category, how many subcategories it has and which ones get the most clicks (with their share of that category's clicks).

In [ ]:
by_sub = shown.groupby(["category", "subcategory"]).agg(shown=("clicked", "size"), clicks=("clicked", "sum")).reset_index()
by_sub["articles"] = by_sub.set_index(["category", "subcategory"]).index.map(news.groupby(["category", "subcategory"]).size())
by_sub["click_share_%"] = (100 * by_sub.clicks / by_sub.clicks.sum()).round(1)
by_sub["share_of_category_%"] = (100 * by_sub.clicks / by_sub.groupby("category").clicks.transform("sum")).round(0)
by_sub["ctr_%"] = (100 * by_sub.clicks / by_sub.shown).round(1)

category_order = by_category.sort_values("clicks", ascending=False).index
by_sub["category"] = pd.Categorical(by_sub["category"], categories=category_order, ordered=True)
by_sub = by_sub.sort_values(["category", "clicks"], ascending=[True, False])

overview = by_sub.groupby("category", observed=True).apply(
    lambda g: pd.Series({
        "subcategories": len(g),
        "category_click_share_%": round(g["click_share_%"].sum(), 1),
        "top_subcategories (share of category)": ", ".join(
            f"{r.subcategory} ({r['share_of_category_%']:.0f}%)"
            for _, r in g.head(5).iterrows()
        ),
    }),
    include_groups=False,
)
pd.set_option("display.max_colwidth", None)
overview

Now the full table: every subcategory with at least 100 clicks, grouped by category (biggest category first).

In [ ]:
pd.set_option("display.max_rows", 200)
by_sub[by_sub.clicks >= 100].set_index(["category", "subcategory"])[
    ["articles", "shown", "clicks", "click_share_%", "share_of_category_%", "ctr_%"]
]

## Step 3: Group categories into 5 buckets

Each bucket matches what a user *means* when they move that slider. Most articles are bucketed by their category; a handful of subcategories are moved because they clearly belong elsewhere.

| Bucket | Means | Categories | Subcategories moved in |
|---|---|---|---|
| News | "What's happening" | news, weather | |
| Learn | "Make me smarter" | finance, health | newsscienceandtechnology (from news) |
| Lifestyle | "My everyday life" | lifestyle, foodanddrink, travel, autos | |
| Entertain | "Fun and celebrities" | entertainment, movies, music, tv, video | lifestyleroyals, lifestylebuzz, lifestylecelebstyle, awardstyle (from lifestyle) |
| Sports | "Games and teams" | sports | |

Why the moves: royals/buzz/celeb style are celebrity gossip, so all celebrity content lives in Entertain (turning Entertain down really removes it). Science and tech is the one part of `news` that is about learning, not current events.

`kids`, `northamerica` and `middleeast` have almost no data, so we drop them.

In [ ]:
CATEGORY_TO_BUCKET = {
    "news": "News", "weather": "News",
    "finance": "Learn", "health": "Learn",
    "lifestyle": "Lifestyle", "foodanddrink": "Lifestyle", "travel": "Lifestyle", "autos": "Lifestyle",
    "entertainment": "Entertain", "movies": "Entertain", "music": "Entertain", "tv": "Entertain", "video": "Entertain",
    "sports": "Sports",
}
SUBCATEGORY_OVERRIDES = {
    "newsscienceandtechnology": "Learn",
    "lifestyleroyals": "Entertain", "lifestylebuzz": "Entertain", "lifestylecelebstyle": "Entertain", "awardstyle": "Entertain",
}
BUCKET_ORDER = ["News", "Learn", "Lifestyle", "Entertain", "Sports"]

def to_bucket(df):
    # Subcategory override wins; otherwise use the category's bucket
    return df["subcategory"].map(SUBCATEGORY_OVERRIDES).fillna(df["category"].map(CATEGORY_TO_BUCKET))

news["bucket"] = to_bucket(news)
shown["bucket"] = to_bucket(shown)

dropped = news[news.bucket.isna()]
print(f"Dropped {len(dropped)} articles from: {sorted(dropped.category.unique())}")

In [ ]:
by_bucket = shown.groupby("bucket").agg(shown=("clicked", "size"), clicks=("clicked", "sum")).reindex(BUCKET_ORDER)
by_bucket.insert(0, "articles", news.groupby("bucket").size())
by_bucket["click_share_%"] = (100 * by_bucket.clicks / by_bucket.clicks.sum()).round(1)
by_bucket["ctr_%"] = (100 * by_bucket.clicks / by_bucket.shown).round(2)
by_bucket

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3.5))
ax.bar(by_bucket.index, by_bucket["click_share_%"], color=["#4C72B0", "#DD8452", "#55A868", "#C44E52", "#8172B2"])
for i, v in enumerate(by_bucket["click_share_%"]):
    ax.text(i, v + 1, f"{v:.0f}%", ha="center")
ax.set_ylabel("Share of all clicks (%)")
ax.set_title("The 'natural' feed mix: what people click without steering")
ax.set_ylim(0, by_bucket["click_share_%"].max() + 10)
ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout()
fig.savefig(FIGURES / "bucket_click_share.png", dpi=150)
plt.show()

## Step 4: Do users have different tastes?

For each user with at least 5 clicks, we work out *their* bucket mix. If everyone looked like the overall average, sliders would be pointless. A wide spread means people really do want different feeds.

In [ ]:
clicks = shown[(shown.clicked == 1) & shown.bucket.notna()]
user_mix = pd.crosstab(clicks.user_id, clicks.bucket, normalize="index").reindex(columns=BUCKET_ORDER, fill_value=0)
active = clicks.user_id.value_counts()
user_mix = user_mix.loc[active[active >= 5].index]

print(f"{len(user_mix):,} users with 5+ clicks")
(100 * user_mix).describe().round(1)

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3.5))
ax.boxplot([100 * user_mix[b] for b in BUCKET_ORDER], tick_labels=BUCKET_ORDER, showfliers=False)
ax.set_ylabel("Share of the user's clicks (%)")
ax.set_title("Each user's bucket mix varies a lot")
ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout()
fig.savefig(FIGURES / "user_bucket_mix.png", dpi=150)
plt.show()

In [ ]:
# How often is one bucket most of a user's clicks? (These are the users a normal feed would "trap".)
top_share = user_mix.max(axis=1)
print(f"Users whose top bucket is >50% of their clicks: {100 * (top_share > 0.5).mean():.0f}%")
print(f"Users whose top bucket is >75% of their clicks: {100 * (top_share > 0.75).mean():.0f}%")
user_mix.idxmax(axis=1).value_counts(normalize=True).mul(100).round(1).rename("users_whose_top_bucket_%")

## Step 5: Is the mix stable day to day?

Our validation set will be the **last day** of train. If the bucket mix jumped around a lot between days, validation would be measuring a different world than training.

In [ ]:
daily = pd.crosstab(clicks.time.dt.date, clicks.bucket, normalize="index").reindex(columns=BUCKET_ORDER).mul(100).round(1)
daily.insert(0, "impressions", behaviors.groupby(behaviors.time.dt.date).size())
daily

## Takeaways

- **The 5 buckets work.** Every bucket has 6,000+ articles. The natural click mix is about **31% News / 14% Learn / 15% Lifestyle / 28% Entertain / 12% Sports**: a normal feed leans toward news and celebrities.
- **Users really do differ.** The middle half of users get 14% to 43% of their clicks from News. For 41% of users, one bucket is more than half of what they click. These are the people a normal feed would "trap".
- **Learn and Lifestyle are the "wish" buckets.** They have the lowest click rates (3.6% and 2.9%), and a quarter of active users never click either one. These are exactly what people say they want more of but don't click, so this is where the stated-vs-revealed preference question will matter most.
- **Sports is all-or-nothing.** Most active users never click sports (median 0%), but the people who do click it a lot, and it has the highest click rate (4.8%).
- **The daily mix wobbles.** Learn ranges from 9% to 20% of clicks depending on the day, and Sports from 8% to 19%. A one-day validation set is fine, but small differences between models on it may be noise.
- **Weekend vs. weekday.** Nov 9–10 (Sat/Sun) have about half the traffic of weekdays.